# Qwen3-4B-Thinking-2507 -- Fine-tune + evaluate on ViNumQA (Unsloth, single Modal session)

Trains `unsloth/Qwen3-4B-Thinking-2507` on ViNumQA and scores PA/EA on `test.json` in ONE notebook --
merged from the separate `qwen3-4b-thinking-2507-stf-w-reasoning-trace.ipynb` +
`qwen3-4b-thinking-2507-eval-only.ipynb` pair, which existed only because Kaggle's 12h session limit
made training + a 497-sample sequential eval loop in one run risky (a measured run spent ~11.2h on
training ALONE). Run on **Modal** instead, where you control session length directly, so that
constraint doesn't apply -- one continuous session trains, saves the adapter, and evaluates it
immediately using the SAME in-memory model (no reload from disk, no Kaggle-dataset upload/attach step
needed for the adapter at all).

**Data**: set `DATASET_VARIANT` in the data-prep cell to pick which dataset to train on. Two are
GOLD-MERGED (every one of the 2993 train samples kept; `reasoning_trace` is null for samples the
teacher did not verifiably solve, trained as bare-program examples with an empty `<think>` block, so
no training data is lost); two are the RAW teacher output, NOT merged -- only the verified samples are
in the file at all, so every training example carries a real reasoning trace and the sample count is
smaller:

| `DATASET_VARIANT` | trace language | verification filter | merged with full split? | train n | valid n |
|---|---|---|---|---|---|
| `v6` | Vietnamese | union (strict OR PA) | yes (2364/2993, 79.0% have a trace) | 2993 | 584 |
| `v6_en` | English | union (strict OR PA) | yes (2187/2993, 73.1% have a trace) | 2993 | 584 |
| `pa` | Vietnamese | PA-scorer only (strictest) | **no** -- 100% of rows have a trace | 2290 | 455 |
| `pa_en` | English | PA-scorer only (strictest) | **no** -- 100% of rows have a trace | 2123 | 450 |

Defaults to `pa` for this Modal run. All four come from the same independent-solve teacher run
(`distill-reasoning-trace/gemma-4-31b-conr-trace-gen-independent-solve*.ipynb`, teacher: gemma-4-31B-it),
where the teacher got only context+question and had to derive the program itself, never the gold
program or answer. No answer-conditioned rationalization pass is included in any variant.

**Same setup as the Kaggle train-only notebook this was merged from** -- `Qwen3-4B-Thinking-2507` is
thinking-only (always emits a `<think>` block, `enable_thinking` is not supported), so the two training
modes are built by hand in `build_conversation` rather than via a template flag, and no literal opening
`<think>` is written (the chat template emits it). Since `pa`/`pa_en` never hit the null-trace branch,
that code path is simply unexercised for those two variants, not removed -- switching back to `v6`/`v6_en`
still works unchanged. `MAX_SEQ_LENGTH = 8192` and the eval loop's `</think>`-based `strip_think` search
by **token id** (`151668`, confirmed single-token for this tokenizer) are both unchanged from the Kaggle
pair -- unlike the Gemma3-4B Modal notebook, which needs a string-based search since `</think>` is not
one token for Gemma's tokenizer.

**Data and outputs**: `test.json` and whichever `{train,valid}_*.json` pair `DATASET_VARIANT` selects are
uploaded manually via the Modal Server Web UI file browser to `/root/`, same pattern as
`distill-reasoning-trace/gemma-4-31b-conr-trace-gen-independent-solve.ipynb` -- for `pa`/`pa_en` this
means the RAW files under `distill-reasoning-trace/outputs/conr_trace_gemma_independent_solve{,_en}/`,
NOT the `datasets/ViNumQA/train_mixed_reasoning_v6*.json` merged ones. The HF cache and the saved
adapter are both pointed at a Modal Volume (`/mnt/qwen3-4b-thinking-2507-sft`) rather than `/root`, so a
container restart doesn't lose the trained adapter -- attach one via the sidebar before running.

**Re-running for a different DATASET_VARIANT on the same Volume**: OUTPUT_DIR/ADAPTER_DIR (training)
and the eval checkpoint files (`eval_partial_k*_<variant>.csv`, `retest_k5_results_<variant>.json`) are
all tagged with `DATASET_VARIANT`, so switching the variable and re-uploading the matching data pair to
`/root/` is enough -- a previous run's adapter and eval checkpoints on the same Volume are never
overwritten or accidentally resumed.

### Installation

In [ ]:
%%capture
# IMPORTANT (real precedent on this repo's Modal setup, not theoretical):
# after this cell finishes, RESTART THE KERNEL (Kernel > Restart) before
# running anything below, then re-run from the top. Modal's Server base image
# ships older typing_extensions/pydantic versions that other packages import
# into the running process BEFORE this cell upgrades them on disk -- the
# upgrade doesn't unload what's already in memory, so `from unsloth import`
# below can fail (e.g. `ImportError: cannot import name 'Sentinel' from
# 'typing_extensions'`) unless the kernel restarts first. This is why `Run
# All` in one uninterrupted pass is not safe here -- run this cell, restart,
# THEN Run All from the top (the pip installs are idempotent, so re-running
# this cell after restart costs a little time, not correctness).
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups (incl. Kaggle)
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install -q tabulate sympy
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import os
from huggingface_hub import login

# Optional: only needed if you plan to push the adapter/merged model to the Hub.
# On Kaggle: Add-ons > Secrets > add "HF_TOKEN", then attach it to this notebook.
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    login(HF_TOKEN)
else:
    print("No HF_TOKEN found -- skipping login (fine unless you want to push_to_hub).")

### HF cache (Modal Volume)

In [ ]:
import os
from pathlib import Path

# Point the HuggingFace cache at the attached Modal Volume instead of the
# container's ephemeral local disk, so the model download only happens once --
# subsequent container restarts load from the Volume. Must run before any
# `from unsloth import ...` / `from transformers import ...` below (env vars
# are read at import time). If your Volume is mounted at a different path,
# update HF_CACHE to match.
HF_CACHE = Path("/mnt/qwen3-4b-thinking-2507-sft")
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hf_cache")

if not HF_CACHE.exists():
    print(f"!! {HF_CACHE} does not exist -- no Volume attached at that path.")
    print("   Attach one via the notebook sidebar (filesystem tab) before continuing,")
    print("   otherwise the model downloads to ephemeral disk and the adapter save")
    print("   below will fail (or be lost on container shutdown).")
else:
    print(f"Volume attached at {HF_CACHE} -- HF cache -> {HF_CACHE / 'hf_cache'}")

### Load base model + LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
import torch

# Sized from the v5 data: the longest fully-formatted training sample is
# ~4.7k tokens (system + context + question + trace + program), so 5678 was
# enough for training. Inference needs the same window to ALSO fit the
# generated answer on top of the prompt, and a 4.7k prompt + 2048 new tokens
# overflows 5678 -- hence 8192. Raising this is cheap: batches are padded to
# their own longest sequence, not to MAX_SEQ_LENGTH.
MAX_SEQ_LENGTH = 8192

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Thinking-2507",
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

In [ ]:
# use_gradient_checkpointing = False (was "unsloth"): trades the memory it saves (by
# recomputing activations in the backward pass) for speed, safe to do on an 80GB card with real
# headroom to spare -- but MAX_SEQ_LENGTH=8192 is long (longer than the Gemma3-4B Modal notebook's
# 5678), so this was NOT numerically verified to fit (no GPU available where this notebook was
# authored). CHECK the "Show current memory stats" cell after the training cell below runs a few
# steps: if peak reserved memory is closing in on 80GB, set this back to "unsloth" and re-run from
# this cell.
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = False,
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

<a name="Data"></a>
### Data prep

Load the official 3-way ViNumQA split and reuse the exact same context formatting (pre_text / markdown
table / post_text) and system prompt as the 0-shot/1-shot/SFT notebooks, so results stay comparable.

In [ ]:
import glob
import pandas as pd
from pathlib import Path
from tabulate import tabulate

# Pick which dataset variant to train on -- everything below (paths, OUTPUT_DIR,
# ADAPTER_DIR) derives from this one switch. Defaults to pa (VI, PA-only, NOT
# merged with the full split) for this Modal run.
#   "v6"    -- VI, union filter (strict OR PA), GOLD-MERGED: every one of the 2993
#              train samples is kept, reasoning_trace is null for the 629 the
#              teacher did not verifiably solve (trained as bare-program). 2364/2993 (79.0%) verified.
#   "v6_en" -- same merge, EN reasoning trace (VI context/program unchanged), union filter. 2187/2993 (73.1%).
#   "pa"    -- VI, PA-scorer-only filter (strictest), NOT merged: only the 2290
#              samples the teacher verifiably solved are in the file at all --
#              no null/bare-program rows, unlike v6/v6_en.
#   "pa_en" -- same as "pa", EN reasoning trace. Only the 2123 verified samples, not merged.
DATASET_VARIANT = "pa"

_VARIANT_FILES = {
    "v6":    ("train_mixed_reasoning_v6.json", "valid_mixed_reasoning_v6.json"),
    "v6_en": ("train_mixed_reasoning_v6_en.json", "valid_mixed_reasoning_v6_en.json"),
    "pa":    ("train_with_reasoning_trace_pa.json", "valid_with_reasoning_trace_pa.json"),
    "pa_en": ("train_with_reasoning_trace_pa_en.json", "valid_with_reasoning_trace_pa_en.json"),
}
TRAIN_FILE, VALID_FILE = _VARIANT_FILES[DATASET_VARIANT]

# Uploaded manually via the Modal Server Web UI file browser -- adjust
# _CANDIDATES if you put the files somewhere else. The two distill-reasoning-trace
# output dirs are only relevant to "pa"/"pa_en" (raw teacher output, un-merged);
# "v6"/"v6_en" live in datasets/ViNumQA instead -- harmless to list all of them,
# since TRAIN_FILE's own name disambiguates which directory actually has it.
_CANDIDATES = [
    Path("/root"),                # Modal: uploaded directly via Server Web UI
    Path("datasets/ViNumQA"),     # local repo path, if running outside Modal
    Path("notebooks/vinumqa/distill-reasoning-trace/outputs/conr_trace_gemma_independent_solve"),
    Path("notebooks/vinumqa/distill-reasoning-trace/outputs/conr_trace_gemma_independent_solve_en"),
]
DATA_DIR = next((p for p in _CANDIDATES if (p / TRAIN_FILE).exists()), None)
if DATA_DIR is None:
    _hits = glob.glob(f"/root/**/{TRAIN_FILE}", recursive=True)
    if _hits:
        DATA_DIR = Path(_hits[0]).parent
if DATA_DIR is None:
    raise FileNotFoundError(
        f"{TRAIN_FILE} not found. Upload {{train,valid}} data for DATASET_VARIANT={DATASET_VARIANT!r} "
        "and test.json via the Server Web UI file browser to /root/, or update _CANDIDATES above."
    )
print(f"Using data from: {DATA_DIR}  (variant={DATASET_VARIANT})")

# test.json is resolved independently of DATA_DIR: for "pa"/"pa_en" the train/valid
# files live in the raw teacher-output dir, which does NOT contain test.json (that
# only lives in datasets/ViNumQA). On Modal this never matters -- everything is
# uploaded flat into /root together -- but resolving it separately keeps the
# notebook correct if the two ever end up in different places.
TEST_DIR = next((p for p in _CANDIDATES if (p / "test.json").exists()), None)
if TEST_DIR is None:
    _hits = glob.glob("/root/**/test.json", recursive=True)
    if _hits:
        TEST_DIR = Path(_hits[0]).parent
if TEST_DIR is None:
    raise FileNotFoundError("test.json not found -- upload it to /root/, or update _CANDIDATES above.")

train_df = pd.read_json(DATA_DIR / TRAIN_FILE)
valid_df = pd.read_json(DATA_DIR / VALID_FILE)
test_df = pd.read_json(TEST_DIR / "test.json")
print(f"train={len(train_df)}, valid={len(valid_df)}, test={len(test_df)}")

n_trace = sum(1 for r in train_df["qa"] if r.get("reasoning_trace"))
print(f"train samples carrying a verified reasoning trace: {n_trace} / {len(train_df)} "
      f"({100 * n_trace / len(train_df):.1f}%)")

In [ ]:
train_df

In [ ]:
def formatting_pre_text(sample):
    return "\n".join(sample["pre_text"])

def formatting_table(sample):
    return tabulate(sample["table"][1:], headers=sample["table"][0], tablefmt="github")

def formatting_post_text(sample):
    return "\n".join(sample["post_text"])

def processing_input_question(sample):
    return sample["qa"]["question"]

def processing_reasoning_trace(sample):
    # .get(), not direct indexing: None for samples without a verified trace,
    # kept as None (not "") so build_conversation can tell the two cases apart.
    return sample["qa"].get("reasoning_trace")

def processing_program_content(sample):
    return sample["qa"]["program"]

def processing_answer_content(sample):
    return sample["qa"]["exe_ans"]

def process_reasoning_split(df):
    df = df.copy()
    df["pre_text_processed"] = df.apply(formatting_pre_text, axis=1)
    df["post_text_processed"] = df.apply(formatting_post_text, axis=1)
    df["table_processed"] = df.apply(formatting_table, axis=1)
    df["table_raw"] = df["table"]  # keep the raw rows for table_* row-name lookup at eval time
    df["input_question"] = df.apply(processing_input_question, axis=1)
    df["reasoning_trace_processed"] = df.apply(processing_reasoning_trace, axis=1)
    df["program_processed"] = df.apply(processing_program_content, axis=1)
    df["answer_processed"] = df.apply(processing_answer_content, axis=1)
    df = df[["pre_text_processed", "table_processed", "table_raw", "post_text_processed", "input_question", "reasoning_trace_processed",
             "program_processed", "answer_processed"]]
    df.columns = ["pre_text", "table", "table_raw", "post_text", "question", "reasoning_trace", "program", "answer"]
    return df

def process_split(df):
    df = df.copy()
    df["pre_text_processed"] = df.apply(formatting_pre_text, axis=1)
    df["post_text_processed"] = df.apply(formatting_post_text, axis=1)
    df["table_processed"] = df.apply(formatting_table, axis=1)
    df["table_raw"] = df["table"]  # keep the raw rows for table_* row-name lookup at eval time
    df["input_question"] = df.apply(processing_input_question, axis=1)
    df["program_processed"] = df.apply(processing_program_content, axis=1)
    df["answer_processed"] = df.apply(processing_answer_content, axis=1)
    df = df[["pre_text_processed", "table_processed", "table_raw", "post_text_processed", "input_question",
             "program_processed", "answer_processed"]]
    df.columns = ["pre_text", "table", "table_raw", "post_text", "question", "program", "answer"]
    return df

train_df = process_reasoning_split(train_df)
valid_df = process_reasoning_split(valid_df)

test_df = process_split(test_df)
test_df["generated_program"] = ""

train_df.sample(n=3)

In [ ]:
SYSTEM_MESSAGE = """You are a financial analysis AI. Your task is to generate a sequential computation program to answer the question, based on the provided context.

### LIST OF 10 VALID OPERATORS:

1. add(a, b) -> a + b
2. subtract(a, b) -> a - b
3. multiply(a, b) -> a * b
4. divide(a, b) -> a / b
5. exp(a, b) -> a^b
6. greater(a, b) -> 1.0 if a > b, else 0.0
7. table_sum(row_name, none) -> sum of the numeric values in the table row named `row_name`
8. table_average(row_name, none) -> arithmetic mean of the numeric values in the table row named `row_name`
9. table_max(row_name, none) -> maximum of the numeric values in the table row named `row_name`
10. table_min(row_name, none) -> minimum of the numeric values in the table row named `row_name`

### RULES:
- Do not use free-form mathematical symbols ("+", "-", "*", "/") outside of parentheses. Every calculation must use one of the 10 operators above.
- table_* operators take exactly two arguments: the row name (copied exactly as it appears as the first cell of the target row) and the literal `none` (e.g. table_max(Lãi ròng, none)), never a list of numeric values.
- Do not perform mental calculations or provide explanations. The output must contain only the program string.
- Reference the result of a previous step using #0 (step 1), #1 (step 2), etc. Steps are separated by commas.
- Preserve the original number format from the context. If a value is missing, use 'none'."""

USER_MESSAGE_FRAME = """### CONTEXT:
[TEXT BEFORE TABLE]
{pre_text}

[TABLE]
{table}

[TEXT AFTER TABLE]
{post_text}

### QUESTION:
{question}

### PROGRAM:"""


### Build the conversational dataset

In [ ]:
# Qwen3-4B-Thinking-2507 is thinking-only: its chat template always opens a
# <think> block and `enable_thinking=False` is not supported (per the model
# card). So the two training modes are built by hand in the assistant content
# rather than via the template flag:
#   - verified trace  -> "{trace}\n</think>\n\n{program}"
#   - no trace (bare) -> "\n</think>\n\n{program}"  (empty think block)
# Neither writes a literal opening <think>; the template emits it, which is
# also why this model's own generations contain only a closing </think>.
def build_conversation(row):
    user_msg = USER_MESSAGE_FRAME.format(
        pre_text=row["pre_text"], table=row["table"],
        post_text=row["post_text"], question=row["question"],
    )

    program = str(row["program"]).strip()
    trace = row["reasoning_trace"]
    # Treat this as a trace ONLY if it is a genuine non-empty string. Missing
    # values arrive as None, float('nan') or pd.NA depending on the pandas
    # version and column dtype, and str() on the latter two yields 'nan' /
    # '<NA>' -- non-empty strings that would silently be trained as the
    # reasoning text. The isinstance check rejects all of them regardless of
    # pandas version. (This bug was real and measured on the non-thinking
    # Qwen3-4B sibling: bare-program rows were training on the literal text
    # "nan" as their <think> content before this guard.)
    if isinstance(trace, str) and trace.strip():
        assistant_content = f"{trace.strip()}\n</think>\n\n{program}"
    else:
        assistant_content = f"\n</think>\n\n{program}"

    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_content},
    ]


def make_text_dataset(df):
    conversations = [build_conversation(row) for _, row in df.iterrows()]
    texts = tokenizer.apply_chat_template(conversations, tokenize=False)
    return texts


train_texts = make_text_dataset(train_df)
valid_texts = make_text_dataset(valid_df)
print(len(train_texts), len(valid_texts))

# Sanity-check the two modes before training: each formatted sample must
# contain exactly one <think> and one </think>. If the counts are off, the
# template is inserting tags differently than assumed above and the assistant
# content needs adjusting -- do not train on it as-is.
_bad = [t for t in train_texts[:200] if t.count("<think>") != 1 or t.count("</think>") != 1]
print(f"malformed think tags in first 200 samples: {len(_bad)} (must be 0)")

_recs = train_df.to_dict("records")
_has = lambda r: isinstance(r["reasoning_trace"], str) and r["reasoning_trace"].strip()
_with = next((i for i, r in enumerate(_recs) if _has(r)), None)
_without = next((i for i, r in enumerate(_recs) if not _has(r)), None)
# "pa"/"pa_en" are un-merged (every row has a real trace), so there is no bare
# example to contrast against -- print whichever of the two exist instead of
# assuming both do (that assumption held for v6/v6_en, not for pa/pa_en).
if _with is not None:
    print("\n===== WITH trace (tail) =====")
    print(train_texts[_with][-900:])
if _without is not None:
    print("\n===== BARE / empty think (tail) =====")
    print(train_texts[_without][-400:])
else:
    print("\n(no bare/empty-think example in this DATASET_VARIANT -- every row has a verified trace)")

In [ ]:
# Compute the max token length across train/valid samples after templating,
# so max_seq_length in SFTConfig can be set to actually cover the longest
# reasoning-trace samples instead of guessing (or silently truncating them).
def compute_token_lengths(texts):
    return [len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in texts]

train_lengths = compute_token_lengths(train_texts)
valid_lengths = compute_token_lengths(valid_texts)

import numpy as np

for name, lengths in [("train", train_lengths), ("valid", valid_lengths)]:
    lengths = np.array(lengths)
    print(f"{name}: n={len(lengths)}  max={lengths.max()}  p99={np.percentile(lengths, 99):.0f}  "
          f"p95={np.percentile(lengths, 95):.0f}  mean={lengths.mean():.0f}")

overall_max = max(max(train_lengths), max(valid_lengths))
print(f"\nOverall max token length: {overall_max}")

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"text": train_texts}).shuffle(seed=3407)
valid_dataset = Dataset.from_dict({"text": valid_texts})
train_dataset, valid_dataset

<a name="Train"></a>
### Train the model

We mask the loss to only the assistant turn (`train_on_responses_only`) so the model isn't penalized for
"predicting" the system prompt / context / question -- standard practice, and important here since the
context block can be much longer than the program string we actually want it to learn to produce.

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = f"/mnt/qwen3-4b-thinking-2507-sft/qwen3-4b-thinking-2507-vinumqa-sft-{DATASET_VARIANT}"  # Volume, not /root -- survives a container restart

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = valid_dataset,
    args = SFTConfig(
        output_dir = OUTPUT_DIR,
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 4,   # was 2 -- A100 80GB has far more VRAM than the original T4 target used
        per_device_eval_batch_size = 4,
        gradient_accumulation_steps = 4,   # was 8 -- effective batch size stays 16, unchanged
        warmup_ratio = 0.03,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 20,
        eval_strategy = "epoch",
        save_strategy = "epoch",
        save_total_limit = 2,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        report_to = "none",
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

In [ ]:
# @title Show current memory stats
import torch
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

Train the model. To resume a run, set `trainer.train(resume_from_checkpoint = True)`.

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference

In [ ]:
FastLanguageModel.for_inference(model)

_row = test_df.iloc[0]
_prompt = USER_MESSAGE_FRAME.format(
    pre_text=_row["pre_text"], table=_row["table"], post_text=_row["post_text"], question=_row["question"],
)
messages = [
    {"role": "system", "content": SYSTEM_MESSAGE},
    {"role": "user", "content": _prompt},
]
text = tokenizer.apply_chat_template(
    # No enable_thinking flag: Thinking-2507 is thinking-only and the model
    # card states the flag is no longer required. Omitting it here keeps the
    # inference formatting byte-identical to how training data was templated.
    messages, tokenize=False, add_generation_prompt=True,
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    # The trained answer (trace + program) tops out near 1k tokens in the v5
    # data, so 2048 is ample; 8192 would have exceeded the context window
    # once added to a long prompt.
    max_new_tokens=2048,
    temperature=0.6, top_p=0.95, top_k=20,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)
print("\nGold program:", _row["program"], "| Gold answer:", _row["answer"])

<a name="Save"></a>
### Saving

Saves LoRA adapters to the attached Modal Volume (not `/root`, which is ephemeral). Uncomment the
`push_to_hub` lines (and set `HF_USERNAME`) to also upload. See the merged-16bit / GGUF cells below
for other export options (same as the reference notebook).

In [ ]:
ADAPTER_DIR = f"/mnt/qwen3-4b-thinking-2507-sft/qwen3-4b-thinking-2507-vinumqa-sft-adapter-{DATASET_VARIANT}"  # Volume, not /root
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Saved LoRA adapter to {ADAPTER_DIR}")

# HF_USERNAME = "your_hf_username"
# repo = f"{HF_USERNAME}/qwen3-4b-thinking-2507-vinumqa-sft-adapter-{DATASET_VARIANT}"
# model.push_to_hub(repo, token=HF_TOKEN)
# tokenizer.push_to_hub(repo, token=HF_TOKEN)

In [ ]:
# Merge to 16bit / 4bit, or export GGUF for llama.cpp -- disabled by default.
if False:
    model.save_pretrained_merged("qwen3-4b-thinking-2507-vinumqa-sft-16bit", tokenizer, save_method="merged_16bit")
if False:
    model.save_pretrained_merged("qwen3-4b-thinking-2507-vinumqa-sft-4bit", tokenizer, save_method="merged_4bit")
if False:
    model.save_pretrained_gguf("qwen3-4b-thinking-2507-vinumqa-sft", tokenizer, quantization_method="q4_k_m")

<a name="Eval"></a>
### PA / EA on the ViNumQA test set

Quick check of Program Accuracy (PA) / Execution Accuracy (EA) right after training, using the
same parser as the 0-shot/1-shot/SFT notebooks so numbers are comparable. Reuses the `model`,
`tokenizer`, `test_df`, `SYSTEM_MESSAGE`, `USER_MESSAGE_FRAME` already built above -- no adapter
reload from disk, no re-fetching test.json. `</think>` IS a single token for Qwen3's tokenizer
(confirmed: id 151668), so the generation loop searches generated-token-ids directly for that id,
same mechanism the Kaggle eval-only notebook uses (unlike the Gemma3-4B Modal notebook, which has
to decode to text first and find the literal substring).

In [ ]:
FastLanguageModel.for_inference(model)  # already called above, harmless to repeat before eval

In [ ]:
# ViNumQA scorer -- kept in sync with notebooks/vinumqa/scorer.py, inlined here
# because a Kaggle notebook cannot import from the repository.
#
"""ViNumQA scorer: the FinQA evaluation protocol, adapted to this dataset.

The shared-task paper states that "the official evaluation protocol proposed by
Chen et al. (2021) is adopted", so the semantics here follow `evaluate/evaluate.py`
(FinQA's own script) rather than being reinvented:

  * Program Accuracy is *symbolic* equivalence, via sympy, between the gold and
    predicted expressions -- not a string or structural match. A prediction may
    reorder or restructure the arithmetic, but it may only use literals that
    appear in the gold program, so it cannot invent constants such as the `100`
    of a percentage rescaling.
  * Execution Accuracy compares the executed result to `exe_ans` exactly, after
    rounding to 5 decimals. No tolerance.
  * `greater` yields the strings "yes"/"no", matching how the dataset stores
    those answers.
  * Every step takes exactly two arguments. Verified against the data: all 663
    steps across the gold programs are binary, and `table_*` always takes a row
    label plus `none` (454 occurrences) rather than a list of values (2).
  * FinQA's `const_` tokens are still understood.

Five corrections are applied, each because the unmodified script cannot
reproduce ViNumQA's own gold, not because the protocol was thought wrong:

1. Tokenisation of bracketed row labels. `program_tokenization` splits on every
   bracket, so `table_min(ROE (%), none)` shatters into six tokens and fails the
   four-tokens-per-step structure check. 35 of the 497 test programs name a row
   whose label contains brackets -- `ROE (%)`, `EPS (VND)`, `P/E (x)` -- and all
   35 were unscoreable. Tokenisation is now bracket-depth aware.

2. Accounting negatives. Tables write negative amounts as `(3344)`. The original
   `process_row` takes the text before the first bracket, leaving an empty
   string, so the cell fails to parse. The dataset's own `exe_ans` was computed
   with those values -- e.g. `table_min(LN hoạt động (tỷ đồng), none)` expects
   -3344 from a row holding `(3344)`. The `-1046 ( 1046 )` form the original
   handled correctly is unchanged.

3. Unparseable cells no longer void the whole row. Measured over the 393 gold
   `table_*(<row>, none)` programs in train, skipping such cells reproduces
   `exe_ans` for 386 against 381 when the row is voided, so skipping is what the
   dataset was built with.

4. `exe_ans` is stored as a string here ("31.0") where FinQA stores a number, so
   the comparison `exe_res == gold_res` was never true. It is coerced, leaving
   the "yes"/"no" answers alone.

5. The `assert exe_res == gold_res` inside the program-accuracy branch is
   dropped. It is a debug check, and a single rounding disagreement aborts the
   whole evaluation.

`evaluate_result_official` runs the unmodified protocol for comparison, so the
cost of each correction can be seen rather than assumed.
"""

import re
from typing import List, Optional, Sequence, Tuple, Union

from sympy import simplify

ALL_OPS = ["add", "subtract", "multiply", "divide", "exp", "greater",
           "table_max", "table_min", "table_sum", "table_average"]

_PAREN_NEG_RE = re.compile(r"^\(\s*([\d.,]+)\s*\)$")
_NAME_RE = re.compile(r"\s*([a-zA-Z_]+)\(")


# ------------------------------------------------------------------ numbers --
def str_to_num(text: str) -> Union[float, str]:
    """FinQA's literal parser, unchanged: returns "n/a" rather than raising."""
    text = str(text).replace(",", "")
    try:
        return float(text)
    except ValueError:
        if "%" in text:
            try:
                return float(text.replace("%", "")) / 100.0
            except ValueError:
                return "n/a"
        if text.endswith(("x", "X")):
            # Multiples are written "14.3x" in these tables. Unlike "%", the
            # suffix carries no scaling -- the gold answer for such a row is the
            # plain multiple.
            try:
                return float(text[:-1])
            except ValueError:
                pass
        if "const" in text:
            text = text.replace("const_", "")
            if text == "m1":
                text = "-1"
            try:
                return float(text)
            except ValueError:
                return "n/a"
        return "n/a"


def _cell_to_num(raw: str) -> Union[float, str]:
    """Parse one table cell.

    Adds the `(3344)` form to what the original handled; `$ -1046 ( 1046 )`
    still resolves through the original's "text before the first bracket" rule.
    """
    text = str(raw).replace("$", "").strip()
    m = _PAREN_NEG_RE.match(text)
    if m:
        value = str_to_num(m.group(1))
        return -value if value != "n/a" else "n/a"
    return str_to_num(text.split("(")[0].strip())


_MISSING_CELL_MARKERS = {"", "-", "–", "—", "na", "n/a", "nan", "none"}


def process_row(row_in: Sequence[str]):
    """Numeric values of a table row, or "n/a" if the row cannot be reduced.

    A cell that merely marks a missing period ("-", "NA", an em dash) is
    skipped: rows in this dataset routinely lack a year or two, and voiding the
    whole row over one gap loses reductions the gold answers depend on. A cell
    with real but unreadable content still voids the row, so genuine parse
    failures are not silently averaged away.
    """
    row_out = []
    for cell in row_in:
        text = str(cell).replace("$", "").strip()
        if text.lower() in _MISSING_CELL_MARKERS:
            continue
        num = _cell_to_num(text)
        if num == "n/a":
            return "n/a"
        row_out.append(num)
    return row_out or "n/a"


# -------------------------------------------------------------- tokenisation --
def program_tokenization(original_program: str) -> List[str]:
    """Tokenise into ['op(', arg1, arg2, ')', ..., 'EOF'].

    Bracket-depth aware, so a row label like `ROE (%)` stays one token. The
    original split on every bracket, which shattered such labels and broke the
    four-tokens-per-step structure the rest of the protocol relies on.

    Raises ValueError if trailing, non-whitespace text remains once no further
    step can be parsed (e.g. a step missing its closing paren, which happens
    both in a handful of gold programs and -- more importantly -- in model
    generations cut off by a max_new_tokens limit). An earlier version of this
    tokenizer silently stopped and returned only the steps parsed so far,
    which let a truncated program like "subtract(100, 50), divide(#0, 5"
    (missing text and closing paren) score as a valid, complete one-step
    program instead of being rejected -- a false positive for exactly the kind
    of generation failure this evaluator needs to catch.
    """
    text = str(original_program).strip()
    program: List[str] = []
    pos = 0

    while pos < len(text):
        m = _NAME_RE.match(text, pos)
        if not m:
            break
        open_idx = m.end() - 1

        depth, close = 0, -1
        for i in range(open_idx, len(text)):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    close = i
                    break
        if close == -1:
            raise ValueError(
                f"Unbalanced parentheses (no matching ')' found) in program: '{original_program}'"
            )

        program.append(m.group(1) + "(")
        # Split arguments on depth-0 commas so brackets inside a label survive.
        args, arg_depth, current = [], 0, []
        for ch in text[m.end():close]:
            if ch == "(":
                arg_depth += 1
                current.append(ch)
            elif ch == ")":
                arg_depth -= 1
                current.append(ch)
            elif ch == "," and arg_depth == 0:
                args.append("".join(current).strip())
                current = []
            else:
                current.append(ch)
        if current:
            args.append("".join(current).strip())

        program.extend(args)
        program.append(")")
        pos = close + 1
        while pos < len(text) and text[pos] in ", ":
            pos += 1

    if pos < len(text) and text[pos:].strip():
        raise ValueError(
            f"Trailing unparsed content in program: '{text[pos:]}' (from: '{original_program}')"
        )

    program.append("EOF")
    return program


def extract_program(raw_text: str) -> str:
    """Recover a program string from raw model output.

    Bracket matched, so an outer call is never silently discarded: the earlier
    regex could only match a bracket-free call, so `multiply(divide(a, b), 100)`
    was reduced to its inner `divide(a, b)` and a percentage rescaling scored as
    if it were the gold answer.
    """
    text = re.sub(r"```[a-zA-Z]*", "", str(raw_text)).replace("```", "").strip()

    calls, pos = [], 0
    while pos < len(text):
        m = _NAME_RE.search(text, pos)
        if not m:
            break
        if m.group(1) not in ALL_OPS:
            pos = m.end()
            continue
        depth, close = 0, -1
        for i in range(m.end() - 1, len(text)):
            if text[i] == "(":
                depth += 1
            elif text[i] == ")":
                depth -= 1
                if depth == 0:
                    close = i
                    break
        if close == -1:
            break
        calls.append(text[m.start(1):close + 1].strip())
        pos = close + 1

    return ", ".join(calls) if calls else text



def _steps_from_tokens(program: List[str]) -> List[Tuple[str, str, str]]:
    """Group a tokenised program into (op, arg1, arg2) triples.

    The original walked the token list by joining it and splitting on ")", which
    silently mis-splits any argument containing a bracket -- exactly the row
    labels this dataset uses, e.g. `EPS (VND)`. Grouping the tokens directly is
    equivalent for well-formed programs and correct for those.
    """
    body = program[:-1] if program and program[-1] == "EOF" else list(program)
    if len(body) % 4 != 0:
        raise ValueError("token count is not a multiple of four")
    steps = []
    for i in range(0, len(body), 4):
        op_token, arg1, arg2, close = body[i:i + 4]
        if not op_token.endswith("(") or close != ")":
            raise ValueError("malformed step")
        op = op_token[:-1].strip()
        if op not in ALL_OPS:
            raise ValueError(f"unknown operator {op!r}")
        steps.append((op, arg1.strip(), arg2.strip()))
    return steps

# ---------------------------------------------------------------- execution --
def eval_program(program: List[str], table: Optional[Sequence[Sequence[str]]]):
    """Execute a tokenised program. Returns (invalid_flag, result)."""
    this_res: Union[float, str] = "n/a"

    try:
        steps = _steps_from_tokens(program)
        res_dict = {}

        for ind, (op, arg1, arg2) in enumerate(steps):
            if op in ("add", "subtract", "multiply", "divide", "exp", "greater"):
                if "#" in arg1:
                    arg1 = res_dict[int(arg1.replace("#", ""))]
                else:
                    arg1 = str_to_num(arg1)
                    if arg1 == "n/a":
                        return 1, "n/a"
                if "#" in arg2:
                    arg2 = res_dict[int(arg2.replace("#", ""))]
                else:
                    arg2 = str_to_num(arg2)
                    if arg2 == "n/a":
                        return 1, "n/a"

                if op == "add":
                    this_res = arg1 + arg2
                elif op == "subtract":
                    this_res = arg1 - arg2
                elif op == "multiply":
                    this_res = arg1 * arg2
                elif op == "divide":
                    this_res = arg1 / arg2
                elif op == "exp":
                    this_res = arg1 ** arg2
                else:
                    this_res = "yes" if arg1 > arg2 else "no"

            else:  # table_*
                table_dict = {row[0]: row[1:] for row in (table or [])}
                if "#" in arg1:
                    num_row = [res_dict[int(arg1.replace("#", ""))]]
                else:
                    if arg1 not in table_dict:
                        return 1, "n/a"
                    num_row = process_row(table_dict[arg1])
                if num_row == "n/a":
                    return 1, "n/a"

                if op == "table_max":
                    this_res = max(num_row)
                elif op == "table_min":
                    this_res = min(num_row)
                elif op == "table_sum":
                    this_res = sum(num_row)
                else:
                    this_res = sum(num_row) / len(num_row)

            res_dict[ind] = this_res

        if this_res not in ("yes", "no", "n/a"):
            this_res = round(this_res, 5)
    except Exception:
        return 1, "n/a"

    return 0, this_res


# ------------------------------------------------------------------ program --
def equal_program(program1: List[str], program2: List[str]) -> bool:
    """Symbolic equivalence of gold (program1) and prediction (program2).

    Same protocol as the official implementation -- literals become symbols,
    table steps become opaque variables, and the two expressions are compared
    after `simplify`, so a differently-arranged but algebraically identical
    program still counts. A prediction may only use symbols that appear in gold,
    which is what stops it from introducing a constant of its own (the `100` of
    a percentage rescaling, say). Only the step-splitting differs: it groups
    tokens rather than splitting a joined string on ")".
    """
    try:
        steps1 = _steps_from_tokens(program1)
    except Exception:
        return False

    sym_map, sym_ind = {}, 0
    for op, arg1, arg2 in steps1:
        if "table" in op:
            key = (op, arg1, arg2)
            if key not in sym_map:
                sym_map[key] = "a" + str(sym_ind)
                sym_ind += 1
        else:
            for arg in (arg1, arg2):
                if "#" not in arg and arg not in sym_map:
                    sym_map[arg] = "a" + str(sym_ind)
                    sym_ind += 1

    try:
        steps2 = _steps_from_tokens(program2)
    except Exception:
        return False

    for ind, (op, arg1, arg2) in enumerate(steps2):
        if "table" in op:
            if (op, arg1, arg2) not in sym_map:
                return False
        else:
            for arg in (arg1, arg2):
                if "#" not in arg:
                    if arg not in sym_map:
                        return False
                elif int(arg.strip("#")) >= ind:
                    return False

    def symbol_recur(ind, steps):
        op, arg1, arg2 = steps[ind]
        if "table" in op:
            return sym_map[(op, arg1, arg2)]
        parts = []
        for arg in (arg1, arg2):
            if "#" in arg:
                parts.append(symbol_recur(int(arg.replace("#", "")), steps))
            else:
                parts.append(sym_map[arg])
        sign = {"add": "+", "subtract": "-", "multiply": "*",
                "divide": "/", "exp": "**", "greater": ">"}[op]
        return f"( {parts[0]} {sign} {parts[1]} )"

    try:
        sym1 = simplify(symbol_recur(len(steps1) - 1, steps1), evaluate=False)
        sym2 = simplify(symbol_recur(len(steps2) - 1, steps2), evaluate=False)
    except Exception:
        return False

    return sym1 == sym2


# ------------------------------------------------------------------ metrics --
def _coerce_answer(value):
    """ViNumQA stores exe_ans as a string; "yes"/"no" stay as they are."""
    try:
        return float(value)
    except (TypeError, ValueError):
        return value


def score_one(generated_program: str, gold_program: str, gold_answer,
              table: Optional[Sequence[Sequence[str]]] = None,
              extract_first: bool = True) -> Tuple[float, float]:
    """(program_accuracy, execution_accuracy) for a single item.

    A generated_program that fails to tokenize (e.g. cut off mid-generation,
    missing a closing paren) scores (0.0, 0.0) rather than raising -- this is
    expected input from a real model, not a bug to surface as an exception.
    gold_program is assumed well-formed and is not caught the same way, so a
    malformed *gold* label still raises loudly instead of silently scoring 0.
    """
    generated = extract_program(generated_program) if extract_first else generated_program
    gold_tok = program_tokenization(gold_program)
    gold_res = _coerce_answer(gold_answer)

    try:
        pred_tok = program_tokenization(generated)
    except ValueError:
        return 0.0, 0.0

    invalid, exe_res = eval_program(pred_tok, table)
    ea = 1.0 if invalid == 0 and exe_res == gold_res else 0.0

    try:
        pa = 1.0 if equal_program(gold_tok, pred_tok) else 0.0
    except Exception:
        pa = 0.0

    return pa, ea


def evaluate_dataframe(df, generated_col: str = "generated_program",
                       gold_program_col: str = "program",
                       gold_answer_col: str = "answer",
                       table_col: str = "table_raw",
                       extract_first: bool = True):
    """Score a DataFrame, returning (df + per-row scores, summary).

    `table_col` must hold the raw table (list of rows); without it, programs
    naming a table row cannot execute and score 0 on EA.
    """
    df = df.copy()
    pa_scores, ea_scores = [], []

    for _, row in df.iterrows():
        table = row[table_col] if table_col in df.columns else None
        pa, ea = score_one(row[generated_col], row[gold_program_col],
                           row[gold_answer_col], table, extract_first)
        pa_scores.append(pa)
        ea_scores.append(ea)

    df["pa_score"] = pa_scores
    df["ea_score"] = ea_scores
    return df, {
        "program_accuracy": sum(pa_scores) / len(pa_scores) if pa_scores else 0.0,
        "execution_accuracy": sum(ea_scores) / len(ea_scores) if ea_scores else 0.0,
    }

In [ ]:
import gc
import glob, os
from pathlib import Path
from tqdm import tqdm

QWEN3_THINK_END_TOKEN_ID = 151668  # "</think>"

# One prompt at a time, deliberately. Batching several prompts together requires
# left padding, and Unsloth's fast inference path derives token positions from
# the cache length rather than from the attention mask -- so padded rows decode
# at the wrong positions. Measured on the wo-reasoning model, that cost 0.6419
# -> 0.6338 PA. Extra votes come from num_return_sequences instead: those share
# a single prompt, so there is nothing to pad, and memory stays bounded by
# N_VOTES rather than by N_VOTES x batch.
#
# N_VOTES = 1 is the like-for-like number. Raising it multiplies the runtime,
# and this model emits a full reasoning trace per sample (~24s each on a T4,
# so ~3.3h for one pass over the 497 test questions) -- k=5 would not fit a
# session. Run k=1 first; the checkpoint below lets a later pass resume.
N_VOTES = 1
# 2048, matching the run this adapter came from. Too low a cap is not a mild
# loss: if generation is cut off before the model closes </think>, strip_think
# finds no closing tag and hands the whole reasoning trace to the parser as if
# it were the program, so a sample that would have been right scores 0 on both
# metrics. Generation stops at EOS anyway, so the higher cap costs nothing on
# samples that finish early.
EVAL_MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.6   # only used when N_VOTES > 1; Qwen3 thinking defaults
TOP_P = 0.95

CKPT_PATH = Path("/mnt/qwen3-4b-thinking-2507-sft") / f"eval_partial_k{N_VOTES}_{DATASET_VARIANT}.csv"  # Volume, tagged with DATASET_VARIANT so switching variants on the same Volume never resumes a different run

if CKPT_PATH.exists():
    _done = pd.read_csv(CKPT_PATH, index_col=0).fillna("")
    test_df.loc[_done.index, "generated_program"] = _done["generated_program"].values
    print(f"Resumed {(test_df['generated_program'] != '').sum()} / {len(test_df)} from working-dir checkpoint.")
else:
    print("No checkpoint found on the Volume -- starting fresh.")


def build_prompt(row):
    user_msg = USER_MESSAGE_FRAME.format(
        pre_text=row["pre_text"], table=row["table"],
        post_text=row["post_text"], question=row["question"],
    )
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_msg},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )


def strip_think(output_ids):
    """Keep only what follows the final </think>; the model always emits one."""
    ids = list(output_ids)
    try:
        cut = len(ids) - ids[::-1].index(QWEN3_THINK_END_TOKEN_ID)
    except ValueError:
        cut = 0
    return tokenizer.decode(ids[cut:], skip_special_tokens=True).strip()


def vote(candidates):
    """Most common program among candidates, keyed by normalised form so that
    cosmetically different but structurally identical programs share a vote."""
    cleaned = [extract_program(c) for c in candidates if c and c.strip()]
    if not cleaned:
        return ""
    keyed = {}
    for c in cleaned:
        try:
            key = str(program_tokenization(c))
        except Exception:
            key = c.strip()
        keyed.setdefault(key, []).append(c)
    best = max(keyed, key=lambda k: (len(keyed[k]), -cleaned.index(keyed[k][0])))
    return keyed[best][0]


n_unclosed = 0
todo = [i for i in test_df.index if not str(test_df.at[i, "generated_program"]).strip()]

for n, df_index in enumerate(tqdm(todo, desc=f"Generating (k={N_VOTES})")):
    enc = tokenizer([build_prompt(test_df.loc[df_index])], return_tensors="pt").to(model.device)

    gen_kwargs = dict(max_new_tokens=EVAL_MAX_NEW_TOKENS)
    if N_VOTES > 1:
        gen_kwargs.update(do_sample=True, temperature=TEMPERATURE, top_p=TOP_P,
                          num_return_sequences=N_VOTES)

    try:
        with torch.no_grad():
            out = model.generate(**enc, **gen_kwargs)
        prompt_len = enc["input_ids"].shape[1]
        gen_only = [r[prompt_len:].tolist() for r in out]
        # A generation that never closed </think> was almost certainly cut off
        # by the token cap; count them so a too-low cap is visible rather than
        # silently scoring zeros.
        n_unclosed += sum(1 for g in gen_only if QWEN3_THINK_END_TOKEN_ID not in g)
        cands = [strip_think(g) for g in gen_only]
        test_df.at[df_index, "generated_program"] = vote(cands) if N_VOTES > 1 else cands[0]
        del enc, out
    except torch.cuda.OutOfMemoryError:
        print(f"  OOM on index {df_index}; leaving it blank (scored 0).")
        test_df.at[df_index, "generated_program"] = ""

    if n % 25 == 0:
        test_df[["generated_program"]].to_csv(CKPT_PATH)
        gc.collect()
        torch.cuda.empty_cache()

test_df[["generated_program"]].to_csv(CKPT_PATH)
print(f"Done. {(test_df['generated_program'] != '').sum()} / {len(test_df)} generated "
      f"(N_VOTES={N_VOTES}, {len(todo)} newly generated this run).")
print(f"Generations that never closed </think>: {n_unclosed} "
      f"-- these were cut off by EVAL_MAX_NEW_TOKENS and score 0; "
      f"raise the cap if this is not near zero.")

In [ ]:
df_scored, summary = evaluate_dataframe(test_df)
print(f"Qwen3-4B-Thinking-2507 (SFT, {DATASET_VARIANT} reasoning trace): PA={summary['program_accuracy']:.4f}  "
      f"EA={summary['execution_accuracy']:.4f}")

### Self-consistency retest on the currently-wrong samples only

Diagnostic, not a full self-consistency run: samples the ~100 samples that already scored
`pa_score == 0 and ea_score == 0` at `df_scored, k=5, do_sample=True` and votes on the
normalised program, then re-scores just that subset. The question this answers is whether
the errors are random (voting recovers a meaningful fraction) or systematic (voting barely
moves the needle) -- run this before committing to a full self-consistency pass over all 497,
or to reworking the training data.

**Not used for reported/production numbers.** Measured on Qwen3-4B-Thinking-2507 (pa_en variant):
k=1 baseline PA 0.7163 / EA 0.7666, full k=5 voting raised this to PA 0.7626 / EA 0.8149 (+4.6/+4.8) --
a real improvement, but rejected for deployment because a live request has no gold label to decide
in advance which queries need the extra votes, so getting this benefit in production would mean
~5x inference cost on every request, not just the ~20% that turn out wrong. Kept here purely as a
diagnostic (random vs systematic errors) for future variants/models, not as a production strategy.

In [ ]:
import gc
import json
from pathlib import Path
from tqdm import tqdm

# Only the samples that are currently wrong on both metrics -- no point
# re-testing the ~395 already-correct ones.
wrong_idx = df_scored.index[(df_scored["pa_score"] == 0) & (df_scored["ea_score"] == 0)].tolist()
print(f"Re-testing {len(wrong_idx)} currently-wrong samples with self-consistency (k=5)...")

N_VOTES_RETEST = 5
TEMPERATURE_RETEST = 0.6
TOP_P_RETEST = 0.95

RETEST_CKPT = Path("/mnt/qwen3-4b-thinking-2507-sft") / f"retest_k5_results_{DATASET_VARIANT}.json"  # tagged with DATASET_VARIANT, same reason as CKPT_PATH above
retest_results = {}
if RETEST_CKPT.exists():
    retest_results = {int(k): v for k, v in json.loads(RETEST_CKPT.read_text(encoding="utf-8")).items()}
    print(f"Resumed {len(retest_results)} / {len(wrong_idx)} from checkpoint.")

todo = [i for i in wrong_idx if i not in retest_results]

for n, df_index in enumerate(tqdm(todo, desc="Self-consistency retest (k=5)")):
    row = test_df.loc[df_index]
    enc = tokenizer([build_prompt(row)], return_tensors="pt").to(model.device)

    try:
        with torch.no_grad():
            out = model.generate(
                **enc, max_new_tokens=EVAL_MAX_NEW_TOKENS,
                do_sample=True, temperature=TEMPERATURE_RETEST, top_p=TOP_P_RETEST,
                num_return_sequences=N_VOTES_RETEST,
            )
        prompt_len = enc["input_ids"].shape[1]
        gen_only = [r[prompt_len:].tolist() for r in out]
        cands = [strip_think(g) for g in gen_only]
        retest_results[df_index] = vote(cands)
        del enc, out
    except torch.cuda.OutOfMemoryError:
        print(f"  OOM on index {df_index}; leaving it blank.")
        retest_results[df_index] = ""

    if n % 10 == 0:
        RETEST_CKPT.write_text(json.dumps(retest_results, ensure_ascii=False), encoding="utf-8")
        gc.collect()
        torch.cuda.empty_cache()

RETEST_CKPT.write_text(json.dumps(retest_results, ensure_ascii=False), encoding="utf-8")
print(f"Done. Retested {len(retest_results)} / {len(wrong_idx)}.")

In [ ]:
retest_df = test_df.loc[wrong_idx].copy()
retest_df["generated_program"] = [retest_results[i] for i in wrong_idx]

retest_scored, retest_summary = evaluate_dataframe(retest_df)
n_fixed = int(((retest_scored["pa_score"] == 1) | (retest_scored["ea_score"] == 1)).sum())

print(f"Self-consistency (k={N_VOTES_RETEST}) on the {len(wrong_idx)} previously-wrong samples:")
print(f"  PA: {retest_summary['program_accuracy']:.4f}   EA: {retest_summary['execution_accuracy']:.4f}")
print()
print(f"  {n_fixed} / {len(wrong_idx)} flipped to correct (PA=1 or EA=1)")
print(f"  -> equivalent to +{100 * n_fixed / len(test_df):.2f} points on the full "
      f"{len(test_df)}-sample PA/EA if these replace the k=1 predictions")
print()
if n_fixed / len(wrong_idx) > 0.15:
    print("A meaningful fraction flipped -- the errors look at least partly random. "
          "Worth running full self-consistency (k=5) over all 497 samples next.")
else:
    print("Barely moved -- the errors look systematic, not sampling noise. "
          "Self-consistency alone is unlikely to help; look at fixing the training "
          "data / reasoning trace instead (see the error-pattern breakdown discussed "
          "earlier).")

In [ ]:
# Final combined result: k=1 baseline for the 381 already-correct samples,
# k=5 self-consistency (majority vote) replacing the predictions for the
# len(wrong_idx) previously-wrong ones -- i.e. exactly what the +X points
# estimate above promised, computed for real instead of estimated.
final_scored = df_scored.copy()
final_scored.loc[retest_scored.index, "pa_score"] = retest_scored["pa_score"]
final_scored.loc[retest_scored.index, "ea_score"] = retest_scored["ea_score"]
final_scored.loc[retest_scored.index, "generated_program"] = retest_scored["generated_program"]

final_pa = final_scored["pa_score"].mean()
final_ea = final_scored["ea_score"].mean()

print(f"Final combined result ({len(test_df) - len(wrong_idx)} kept from k=1 + {len(wrong_idx)} replaced by k=5 vote):")
print(f"  PA: {final_pa:.4f}  (k=1 baseline {summary['program_accuracy']:.4f} -> {'+' if final_pa >= summary['program_accuracy'] else ''}{final_pa - summary['program_accuracy']:.4f})")
print(f"  EA: {final_ea:.4f}  (k=1 baseline {summary['execution_accuracy']:.4f} -> {'+' if final_ea >= summary['execution_accuracy'] else ''}{final_ea - summary['execution_accuracy']:.4f})")

## Done

The LoRA adapter is saved under `/mnt/qwen3-4b-thinking-2507-sft/qwen3-4b-thinking-2507-vinumqa-sft-adapter-<DATASET_VARIANT>`,
and PA/EA on the full test set is printed above -- both in this one session, no hand-off between
notebooks needed.